<div
<br><br>
<h1 style="color:#7F000E;"> DMC </h1>
<h1 style="color:#7F000E;"> Aprendizaje no Supervisado y algoritmos de clusterización </h1>

<h3 style="color:#7F000E;"> Métodos No Supervisados </h3>
<h3 style="color:#7F000E;"> Algoritmo DBSCAN </h3>
</div>
<br><br>
<div style="text-align:right">

<span style="color:#7F000E; font-size:14px;">Ing. César Quezada</span><br>
<span style="color:#7F000E; font-size:14px;">Horario: 19:00 – 22:00</span><br>
<span style="color:#7F000E; font-size:14px;">Sesión 02</span>

</div>


#### Caso:

Dado que los clientes reciben diversas ofertas de consumo para que estos puedan transaccionar con su tarjeta de crédito y débito, la entidad bancaria no está segura si sus clientes tienen algún interes en sus ofertas, teniendo en cuenta que cada oferta ya corresponde un gasto para la entidad.

Por tanto: Se pide realizar un estudio de segmentación para conocer cuáles son las preferencias de consumo que sus clientes optarían para comunicarles ofertas más direccionadas.

### 1. Librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (7, 4)
plt.style.use('ggplot')

### 2. Extracción Base de datos

In [ ]:
dataFramePre = pd.read_csv("02dataBaseConsumo.txt",delimiter='|', encoding='latin-1')
dataFramePre.head()

In [ ]:
dataFramePre.info()

In [ ]:
# Convertir a categórica
dataFramePre['flgLimaProv'] = dataFramePre['flgLimaProv'].astype('category')
# etiquetar categorías
dataFramePre['flgLimaProv'] = dataFramePre['flgLimaProv'].cat.rename_categories({1: 'Lima', 0: 'Provincia'})

In [ ]:
dataFramePre.head()

In [ ]:
freq_trx = dataFramePre.groupby(['cliente']).size().reset_index(name='freq_trx')
freq_trx

### 3. Metodología

In [ ]:
#### 3.1 Análisis Previo (objetivo)
#### 3.2 Exploración (descriptivo, grafico barras,cajas)
#### 3.3 Transformación (standarización,cajas)
#### 3.4 Outliers (analisis y eliminación de outliers)
#### 3.5 Dimensionamiento (PCA)
#### 3.6 Modelamiento
#### 3.7 Evaluación
#### 3.8 Perfilamiento
#### 3.9 Visualización

#### 3.1 Análisis Previo

Qué clase de rubros de consumo tenemos?

In [ ]:
copy = pd.DataFrame()
rubroResum = pd.DataFrame()

copy = dataFramePre.copy()
rubroResum["ctdTrx"] = copy.groupby("grupoGiro").agg("trx").sum()

print('Cantidad de Rubros: '+str(rubroResum.shape[0]))
print('\n')
print(rubroResum.sort_values("ctdTrx",ascending=False))

In [ ]:
cantidadGrupo =  pd.DataFrame()
cantidadGrupo['ctdCliente'] = copy.groupby('codmes').agg('cliente').nunique()
cantidadGrupo['ctdTrx']= copy.groupby('codmes').agg('trx').sum()
cantidadGrupo['promTrxCli'] = round(cantidadGrupo['ctdTrx']/cantidadGrupo['ctdCliente'],2)

# Asegurar orden cronológico de los meses
cantidadGrupo = cantidadGrupo.sort_index()

cantidadGrupo

In [ ]:
# Convertir meses a string para matplotlib
meses = cantidadGrupo.index.astype(str)


# ==============================
# GRAFICO 1: TRANSACCIONES TOTALES
# ==============================

plt.figure()  # tamaño default
plt.bar(meses, cantidadGrupo['ctdTrx'], color='blue')
plt.title('Transacciones Históricas')
plt.xlabel('Mes')
plt.ylabel('Cantidad de Transacciones')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# ==============================
# GRAFICO 2: PROMEDIO POR CLIENTE
# ==============================

plt.figure()  # nueva figura independiente
plt.bar(meses, cantidadGrupo['promTrxCli'], color='red')
plt.title('Transacciones Históricas Promedio por Cliente')
plt.xlabel('Mes')
plt.ylabel('Prom Trx por Cliente')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Qué valores es recomnedable usar para el estudio?

In [ ]:
cantidadGrupo  =  pd.DataFrame()
cantidadGrupo2 =  pd.DataFrame()

cantidadGrupo['frecMeses'] = copy.groupby('cliente').agg('codmes').nunique()
cantidadGrupo['sumTrx'] = copy.groupby('cliente').agg('trx').sum()
cantidadGrupo

In [ ]:
cantidadGrupo2['ctdCliente'] = cantidadGrupo.groupby('frecMeses').size()
cantidadGrupo2['ctdTrx'] = cantidadGrupo.groupby('frecMeses').agg('sumTrx').sum()
cantidadGrupo2.head(12)

In [ ]:
cantidadGrupo2['ctdTrxMes'] = round(cantidadGrupo2['ctdTrx']/cantidadGrupo2.index,2)
cantidadGrupo2['promTrxCli'] = round(cantidadGrupo2['ctdTrxMes']/cantidadGrupo2['ctdCliente'],2)
cantidadGrupo2

In [ ]:
# ==============================
# GRAFICO DE LINEAS:
# ==============================

plt.plot(cantidadGrupo2['promTrxCli'])
plt.xlim(1,12)
plt.ylim(0,6)
plt.title('Autoasignados')
plt.show()
print(cantidadGrupo2)

Luego del análisis de datos con respecto a la transacción en su historia, iniciamos la construcción de nuestra Matriz de Segmentación. Por tanto llevaremos nuestra base de datos a nivel de "cliente" y creando variables de rubro de consumo.

In [ ]:
# Reinicio de índice "df.reset_index()"
copy=pd.DataFrame()
copy=dataFramePre.copy()
dataFrame = pd.DataFrame()

dataFrame['trxGrupoGiro']=copy.groupby(["cliente","grupoGiro","edad","ingreso","sexo"]).agg("trx").sum()
dataFrame = pd.pivot_table(dataFrame,'trxGrupoGiro',['cliente',"edad","ingreso","sexo"],'grupoGiro')
dataFrame = dataFrame.fillna(0)

In [ ]:
dataFrame.head()

In [ ]:
dataFrame = dataFrame.reset_index()
cantidadGrupo = cantidadGrupo.reset_index()

dataFrame["frecMeses"] = cantidadGrupo["frecMeses"]
dataFrame.head()

In [ ]:
# Eliminando los valores autoasignados por no ser estables:
dataFrame = dataFrame[dataFrame["frecMeses"]>=5]
dataFrame = dataFrame[dataFrame["frecMeses"]<=10]
dataFrame.head()

In [ ]:
# Variables objetivo de estudio:
rubroName = ['prodsuper', 'restbar', 'salud', 'vehrep', 'entretenimiento', 'tiendadepar', 'ropamoda', 'prodpersondiv',
               'telcom','financiero', 'transplaerea','clubmkt','prodlocal','enseñanza','belleza','prodelectro',
               'alqbienes','artcultura','profdiverso','hogaroficina','informatica']

In [ ]:
print("Número de filas: " + str(dataFrame.shape[0]))
print("Número de columnas: " + str(dataFrame.shape[1]))

#### 3.2 Exploración

In [ ]:
pd.options.display.max_columns = None
dataFrame[rubroName].describe()

In [ ]:
dataFrame[rubroName].hist(bins = 100, figsize=(20,15))
plt.show()

In [ ]:
# Gráfico de cajas por variable en estudio:
for columnName in rubroName:
    plt.title(columnName)
    plt.boxplot(dataFrame[columnName], 0, 'gD')
    plt.show()

In [ ]:
skew_channels = dataFrame[rubroName].skew()
kurt_channels = dataFrame[rubroName].kurtosis()

skew_channels, kurt_channels

Distribución y sparsity (clientes sin uso del canal)

In [ ]:
# Esto te dice qué rubros son masivos vs nicho
# ---
zero_rate = (dataFrame[rubroName] == 0).mean().sort_values(ascending=False)
zero_rate

In [ ]:
# Correlaciones >0.9, se debe considerar PCA o eliminar variables redundantes
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))
sns.heatmap(dataFrame[rubroName].corr(), annot=True, cmap="coolwarm")
plt.title("Correlación entre rubros")
plt.show()

#### 3.3 Transformación

In [ ]:
# ------------------------------
# Creamos el objeto para escalar
# ------------------------------
from sklearn import preprocessing

scaler = preprocessing.MinMaxScaler()


# Guardamos el dataframe original
df_real = dataFrame.copy()

# ************
# Lo aplicamos
# ************
for columnName in rubroName:
    dataFrame[columnName] = scaler.fit_transform(dataFrame[columnName].values.reshape(-1, 1))

In [ ]:
dataFrame[rubroName].head()

#### 3.4 Outliers

In [ ]:
# Cálculo de intervalo del diagrama de cajas - Método de Rango Intercuartílico
#def calculateNumOutliars(serie):
#  Q01 = serie.quantile(0.25)
#  Q03 = serie.quantile(0.75)
#  IQR = Q03 - Q01
#  a = (serie < (Q01 - 1.5 * IQR)) | (serie > (Q03 + 1.5 * IQR))
#  numOutliars = a[a == True].shape[0]
#  return numOutliars

In [ ]:
# Usamos el método de Z-score (considerando se distribuye Normalmente) --- para grandes volúmenes de datos
def calculateNumOutliars(serie):
    mu = serie.mean()
    desv = np.std(serie)
    a = ((serie-mu)/desv < -2) | ((serie-mu)/desv > 2)
    numOutliars = a[a == True].shape[0]
    return a,numOutliars

In [ ]:
numTotal = dataFrame.shape[0]
for columnName in rubroName:
    a,numOutliars = calculateNumOutliars(dataFrame[columnName])
    # Creamos nuevos campos para filtrar los Outliers
    dataFrame['flg_'+columnName]=a
    print('*'+columnName)
    if numOutliars > 0:
      print("Número de valores outliars: " + str(numOutliars))
      print("Porcentaje: " + str(np.round(numOutliars * 100 / numTotal, 2)) + "%")
    else:
      print("****No hay Outliers")
    print("\n")

In [ ]:
# ************************
# Extrayendo los Outliers
# ************************
# Luego que cada variable tenga menos del 10% de Outlier, se filtra de manera Multivariada (este filtro podría ser
# considerado como un segmento Heavy)

dataFrameClear = dataFrame[(dataFrame['flg_prodsuper']==False)&
                      (dataFrame['flg_restbar']==False)&
                      (dataFrame['flg_salud']==False)&
                      (dataFrame['flg_vehrep']==False)&
                      (dataFrame['flg_entretenimiento']==False)&
                      (dataFrame['flg_tiendadepar']==False)&
                      (dataFrame['flg_ropamoda']==False)&
                      (dataFrame['flg_prodpersondiv']==False)&
                      (dataFrame['flg_telcom']==False)&
                      (dataFrame['flg_financiero']==False)&
                      (dataFrame['flg_transplaerea']==False)&
                      (dataFrame['flg_clubmkt']==False)&
                      (dataFrame['flg_prodlocal']==False)&
                      (dataFrame['flg_enseñanza']==False)&
                      (dataFrame['flg_belleza']==False)&
                      (dataFrame['flg_prodelectro']==False)&
                      (dataFrame['flg_alqbienes']==False)&
                      (dataFrame['flg_artcultura']==False)&
                      (dataFrame['flg_profdiverso']==False)&
                      (dataFrame['flg_hogaroficina']==False)&
                      (dataFrame['flg_informatica']==False)]



# GUARDAR LOS ÍNDICES EN UNA LISTA
idx_filtrados = dataFrameClear.index.tolist()

# Refrescamos los índices del Data frame final
# dataFrameClear = dataFrameClear.reset_index()

print('Cantidad de Registros sin Outliers: '+str(dataFrameClear.shape[0]))
dataFrameClear[rubroName].head()

In [ ]:
# Asignación DataFrame:
df = pd.DataFrame()
df = dataFrameClear.copy()

In [ ]:
# Gráfico de cajas por variable en estudio:
for columnName in rubroName:
    plt.title(columnName)
    plt.boxplot(df[columnName], 0, 'gD')
    plt.show()

In [ ]:
rubroName_fin = ['prodsuper','restbar','salud','entretenimiento']

In [ ]:
df.head()

#### 3.6 Modelamiento

##### 3.6.1. KMEANS

In [ ]:
from sklearn.cluster import KMeans
from sklearn import metrics

In [ ]:
# Calculando el número de clúster adecuado:
X = df[rubroName_fin].copy()

numClus = range(1, 20)
kmeans = [KMeans(n_clusters=i,max_iter=600) for i in numClus]
kmeans
score = [kmeans[i].fit(X).score(X) for i in range(len(kmeans))]
score
plt.plot(numClus,score)
plt.xlabel('Número de Clúster')
plt.ylabel('Score')
plt.title('Curva de Inflexión')
plt.show()

In [ ]:
# Nos fijamos de los indicadores de clustering:

ctdDf = int(0.1*dataFrame.shape[0])
cluster = [kmeans[i].predict(X) for i in range(len(kmeans))]

for i in range(1,11):
    print(str(i+1)+' clústeres:')
    print('Inercia: '+str(kmeans[i].inertia_))
    print('Silueta: '+str(metrics.silhouette_score(X, cluster[i], metric='euclidean',sample_size=ctdDf)))
    print("\n")

In [ ]:
numClus = [3,4,5,6,7]

centroide = [kmeans[i].cluster_centers_ for i in range(len(kmeans))]
copy =  pd.DataFrame()

for i in numClus:
    # Distribución de los grupos por clúster:
    copy['cluster'] = cluster[i-1]
    cantidadGrupo =  pd.DataFrame()
    cantidadGrupo['ctdCliente']=copy.groupby('cluster').size()
    cantidadGrupo['pctCliente']=round(100*cantidadGrupo['ctdCliente']/cantidadGrupo['ctdCliente'].sum(),2)

    # gráfico de los grupos según su distribución:
    plt.pie(cantidadGrupo['pctCliente'], labels=cantidadGrupo.index, autopct='%1.1f%%')
    plt.title('Clúster '+str(i))
    plt.legend()
    plt.show()
    print(cantidadGrupo)
    print('\n')

In [ ]:
df_real = df_real.loc[idx_filtrados]

df_ = pd.DataFrame()
df_ = df_real.copy()

In [ ]:
nCluster = int(input('Ingrese la cantidad de cluster: '))
df_['cluster_kmeans'] = cluster[nCluster-1]

In [ ]:
df_.head()

##### 3.6.2. DBSCAN

In [ ]:
# Calculando el número de clúster adecuado:
X = df[rubroName_fin]
X

Hallando los 2 Hiperparámetros tentativos:

In [ ]:
from sklearn.neighbors import NearestNeighbors

neighb = NearestNeighbors(n_neighbors=4)
nbrs = neighb.fit(X)
# Encontrando la distancia del vecino más cercana
distances,indices=nbrs.kneighbors(X)

In [ ]:
# Visualizando la distancia de resultado

#Ordenando las distancias
dist = np.sort(distances, axis = 0)
dist = dist[:, 1]

plt.rcParams['figure.figsize'] = (8,5)
plt.plot(dist)
plt.show()

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn import metrics

In [ ]:
# Número de ountos mínimos (muestra)
numSam = range(120, 160)
dbscan = [DBSCAN(eps=0.005, min_samples=i) for i in numSam]
dbscan
modDbscan = [dbscan[i].fit(X) for i in range(len(dbscan))]
modDbscan
cluster = [modDbscan[i].labels_ for i in range(len(dbscan))]
silhou  = [metrics.silhouette_score(X, cluster[i]) for i in range(len(dbscan))]

plt.plot(numSam,silhou)
plt.xlabel('Muestra')
plt.ylabel('Silueta')
plt.title('Parámetro DBSCAN')
plt.show()

In [ ]:
# Nos fijamos de los indicadores de Clustering:

for i in range(len(numSam)):
    print('* Posición Nro: '+str(i+1))
    print(str(numSam[i])+' puntos mínimos:')
    print('Ctd clúster: '+str(len(set(cluster[i]))))
    print('Silueta: '+str(silhou[i]))
    print("\n")

Visualizando los grupos en 2-D para tener alguna noción de como se agrupan, en esta ocasión probaremos distintos par de variables

In [ ]:
# Vector de colores según el clúster

orden = 5
uniqueClus = set(cluster[orden-1])
colors = [plt.cm.Spectral(each)
          for each in np.linspace(0, 1, len(uniqueClus))]

In [ ]:
fig = plt.figure()
f1 = df['prodsuper'].values
f2 = df['salud'].values

colores=colors
asignar=[]
for row in cluster[orden-1]:
    asignar.append(colores[row])

plt.scatter(f1, f2, c=asignar, s=900)
plt.show()

In [ ]:
fig = plt.figure()
f1 = df['entretenimiento'].values
f2 = df['salud'].values

colores=colors
asignar=[]
for row in cluster[orden-1]:
    asignar.append(colores[row])

plt.scatter(f1, f2, c=asignar, s=1300)
plt.show()

#### 3.7 Evaluación

In [ ]:
ordenMu = [1,14,18]

In [ ]:
copy =  pd.DataFrame()

for i in ordenMu:
    # Distribución de los grupos por clúster:
    copy['cluster'] = cluster[i-1]

    cantidadGrupo =  pd.DataFrame()
    cantidadGrupo['ctdCliente']=copy.groupby('cluster').size()
    cantidadGrupo['pctCliente']=round(100*cantidadGrupo['ctdCliente']/cantidadGrupo['ctdCliente'].sum(),2)

    # gráfico de los grupos según su distribución:
    plt.pie(cantidadGrupo['pctCliente'], labels=cantidadGrupo.index, autopct='%1.1f%%')
    plt.title('Clúster '+ str(len(set(cluster[i-1]))))
    plt.show()
    print(cantidadGrupo)
    print('\n')

#### 3.8 Perfilamiento

In [ ]:
ordenCluster = int(input('Ingrese el orden de clúster: '))

In [ ]:
# Pegamos los datos de los clústering:
df_['clusterDBSCAN'] = cluster[ordenCluster-1]

In [ ]:
df_.head()

In [ ]:
resClus = df_.groupby('clusterDBSCAN').agg({'cliente':'count','restbar':'mean','salud':'mean',
                                     'prodsuper':'mean','entretenimiento':'mean'}).sort_values(by='clusterDBSCAN')

resClus = resClus.reset_index()
resClus['%'] = round(100*resClus['cliente']/copy.count()[0],1)
print(f'Clientes Total: {copy.count()[0]}\n')
resClus